# Colab notebook → professional PDF

**How to use**

1. In the notebook you want to convert: **File ▸ Download ▸ Download .ipynb** (run it first so the outputs are saved).
2. Press ▶ on the cell below and choose that file (you can pick several at once).
3. The PDF downloads automatically. The first run of a session installs a headless browser, which takes a little while; later runs take seconds.

**Options** (on the right of the cell): paper size, landscape for wide tables, and *HIDE_CODE* for a report with outputs only.


In [ ]:
#@title 📄 Colab notebook → PDF — press ▶, then choose your .ipynb { display-mode: "form" }
#@markdown Leave **NOTEBOOK_PATH** empty to upload from your computer, or paste a Drive path such as
#@markdown `/content/drive/MyDrive/Colab Notebooks/HW2.ipynb` to skip the download/upload step.
NOTEBOOK_PATH = ""      #@param {type:"string"}
PAGE_SIZE = "Letter"    #@param ["Letter", "A4"]
LANDSCAPE = False       #@param {type:"boolean"}
HIDE_CODE = False       #@param {type:"boolean"}

import asyncio, html, importlib, importlib.util, os, subprocess, sys, tempfile
from pathlib import Path
from google.colab import drive, files

# ── 1. Start installing the PDF engine in the background (only the first time per session) ──
SETUP_LOG = Path(tempfile.gettempdir()) / "pdf_setup.log"
BROWSERS = Path(os.environ.get("PLAYWRIGHT_BROWSERS_PATH", Path.home() / ".cache/ms-playwright"))

def engine_ready():
    return (all(importlib.util.find_spec(m) for m in ("playwright", "nbconvert"))
            and any(BROWSERS.glob("chromium*/INSTALLATION_COMPLETE")))

setup = None
if not engine_ready():
    py = f'"{sys.executable}" -m'
    setup = subprocess.Popen(
        f"{py} pip install -q playwright 'nbconvert>=7' && {py} playwright install --with-deps chromium",
        shell=True, stdout=SETUP_LOG.open("w"), stderr=subprocess.STDOUT)

# ── 2. Collect the notebook(s) while the install runs ──
notebooks = {}
if NOTEBOOK_PATH.strip():
    path = Path(NOTEBOOK_PATH.strip())
    if str(path).startswith("/content/drive") and not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    if not path.is_file():
        raise FileNotFoundError(f"No notebook found at: {path}")
    notebooks[path.name] = path.read_bytes()
else:
    print("Choose one or more .ipynb files (File ▸ Download ▸ Download .ipynb in the notebook you want):")
    notebooks = {name: data for name, data in files.upload().items()}

if setup is not None:
    print("⏳ Setting up the PDF engine (first run of the session only)…")
    if setup.wait() != 0:
        raise RuntimeError("PDF engine setup failed:\n" + SETUP_LOG.read_text()[-3000:])
    importlib.invalidate_caches()

import nbformat
from nbconvert import HTMLExporter
from playwright.async_api import async_playwright

# ── 3. Notebook → HTML (exactly what the browser shows: plots, tables, math, colours) ──
PRINT_CSS = """
html, body, .jp-Notebook { margin: 0 !important; padding: 0 !important; }
body { -webkit-print-color-adjust: exact; print-color-adjust: exact; }
/* never lose text off the right edge: wrap long code / output lines like Colab does */
pre, code, .highlight pre { white-space: pre-wrap !important; overflow-wrap: anywhere !important; }
.jp-OutputArea-output, .jp-RenderedHTMLCommon, .jp-InputArea-editor { overflow: visible !important; }
.jp-RenderedHTMLCommon { overflow-wrap: break-word; }
/* keep figures and table rows whole, keep headings attached to what follows */
img { max-width: 100% !important; height: auto !important; }
img, svg, canvas, tr, .MathJax_Display, .plotly-graph-div { break-inside: avoid; }
h1, h2, h3, h4, h5, h6 { break-after: avoid; }
.jp-OutputArea-child { break-inside: auto; }
.jp-RenderedHTMLCommon table { font-size: 0.85em; }
.jp-RenderedMarkdown table { margin-left: 0 !important; }
/* hide Colab's interactive-only buttons that are saved inside DataFrame outputs */
.colab-df-buttons, .colab-df-spinner, button[class^="colab-df"], div[id^="id_"] > button { display: none !important; }
"""

def notebook_to_html(raw, title):
    nb = nbformat.reads(raw.decode("utf-8"), as_version=4)
    nb.metadata.pop("widgets", None)  # Colab's saved widget state crashes nbconvert ("KeyError: 'state'")
    for cell in nb.cells:
        for out in cell.get("outputs", []):
            data = out.get("data", {})
            if len(data) > 1:  # show a widget's text fallback (e.g. tqdm bar) instead of an empty box
                data.pop("application/vnd.jupyter.widget-view+json", None)
    nb.cells = [c for c in nb.cells if c.source.strip() or c.get("outputs")]  # drop empty cells
    exporter = HTMLExporter(template_name="lab", embed_images=True, exclude_input=HIDE_CODE,
                            exclude_input_prompt=True, exclude_output_prompt=True)
    body, _ = exporter.from_notebook_node(nb, resources={"metadata": {"name": title}})
    return body.replace("</head>", f"<style>{PRINT_CSS}</style></head>", 1)

# ── 4. HTML → PDF with headless Chromium ──
WAIT_FOR_RENDER = """() => new Promise(done => {
    setTimeout(done, 20000);                                   // hard cap so we can never hang
    const fonts = document.fonts ? document.fonts.ready : Promise.resolve();
    fonts.then(() => {
        if (window.MathJax && MathJax.Hub) MathJax.Hub.Queue(() => done());                     // MathJax 2
        else if (window.MathJax && MathJax.startup) MathJax.startup.promise.then(() => done()); // MathJax 3
        else done();
    });
})"""

# Colab lets wide tables scroll sideways; paper can't, so shrink anything wider than the page to fit.
FIT_TO_PAGE = """() => {
    let smallest = 1;
    const fit = (el, available) => {
        const needed = Math.max(el.scrollWidth, el.offsetWidth);
        if (available > 0 && needed > available + 1) {
            el.style.zoom = available / needed;
            smallest = Math.min(smallest, available / needed);
        }
    };
    for (const out of document.querySelectorAll('.jp-OutputArea-output')) fit(out, out.clientWidth);
    for (const cell of document.querySelectorAll('.jp-RenderedMarkdown:not(.jp-OutputArea-output)')) {
        const cs = getComputedStyle(cell);   // text cells: shrink only the wide table / equation, not the text
        const inner = cell.clientWidth - parseFloat(cs.paddingLeft) - parseFloat(cs.paddingRight);
        for (const child of cell.children) fit(child, inner);
    }
    return smallest;
}"""
MARGIN_IN = 0.6
PAPER_IN = {"Letter": (8.5, 11.0), "A4": (8.27, 11.69)}

async def html_to_pdf(browser, html_text, pdf_path, title):
    with tempfile.NamedTemporaryFile("w", suffix=".html", delete=False, encoding="utf-8") as f:
        f.write(html_text)
    paper_width = PAPER_IN[PAGE_SIZE][1 if LANDSCAPE else 0]
    page = await browser.new_page(viewport={"width": round((paper_width - 2 * MARGIN_IN) * 96), "height": 1000})
    try:
        await page.emulate_media(media="print")  # lay out exactly as it will be printed
        await page.goto(Path(f.name).as_uri(), wait_until="load", timeout=120000)
        try:  # let MathJax / Plotly / remote images finish loading
            await page.wait_for_load_state("networkidle", timeout=30000)
        except Exception:
            pass
        await page.evaluate(WAIT_FOR_RENDER)
        await page.wait_for_timeout(300)
        if await page.evaluate(FIT_TO_PAGE) < 0.7 and not LANDSCAPE:
            print(f"ℹ️ {title}: a very wide table was shrunk to fit the page — tick LANDSCAPE for larger text.")
        footer = (f'<div style="font-size:8px;color:#777;width:100%;padding:0 {MARGIN_IN}in;display:flex;'
                  f'justify-content:space-between;"><span>{html.escape(title)}</span>'
                  '<span>Page <span class="pageNumber"></span> of <span class="totalPages"></span></span></div>')
        await page.pdf(path=str(pdf_path), format=PAGE_SIZE, landscape=LANDSCAPE, print_background=True,
                       display_header_footer=True, header_template="<span></span>", footer_template=footer,
                       margin={"top": f"{MARGIN_IN}in", "bottom": "0.7in", "left": f"{MARGIN_IN}in", "right": f"{MARGIN_IN}in"},
                       outline=True, tagged=True)  # outline = clickable PDF bookmarks from your headings
    finally:
        await page.close()
        os.unlink(f.name)

# ── 5. Convert everything and download ──
done = []
async with async_playwright() as pw:
    browser = await pw.chromium.launch(args=["--no-sandbox"])
    for name, raw in notebooks.items():
        if not name.lower().endswith(".ipynb"):
            print(f"⚠️ Skipped {name} (not a .ipynb file)")
            continue
        try:
            pdf_path = Path(name).with_suffix(".pdf")
            await html_to_pdf(browser, notebook_to_html(raw, Path(name).stem), pdf_path, Path(name).stem)
            print(f"✅ {name} → {pdf_path}  ({pdf_path.stat().st_size / 1e6:.1f} MB)")
            done.append(pdf_path)
        except Exception as err:
            print(f"❌ {name} could not be converted: {type(err).__name__}: {err}")
    await browser.close()

for pdf_path in done:
    files.download(str(pdf_path))
if not notebooks:
    print("No file was selected — run the cell again and pick a .ipynb file.")